**ROI IDENTIFICATON**

Removes any false positives by morphological (pixels) and functional (snr) characteristics in the suite2p output file. It assumes all ROIs are pre-selected in suite2p

(Elongation and compactness not used as that will exclude axonal boutons)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from scipy import ndimage, stats
from skimage.measure import regionprops
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.style.use('default')

# ============================================================================
# CHUNK 1: DATA LOADING & SETUP
# ============================================================================

def load_suite2p_data(suite2p_path):
    """
    Load all necessary Suite2p files
    
    Parameters:
    -----------
    suite2p_path : str or Path
        Path to suite2p/plane0 folder
    
    Returns:
    --------
    data_dict : dict
        Dictionary containing stat, iscell, ops, F, and mean_img
    """
    suite2p_path = Path(suite2p_path)
    
    # Check if path exists
    if not suite2p_path.exists():
        raise FileNotFoundError(f"Suite2p path not found: {suite2p_path}")
    
    print("Loading Suite2p files...")
    
    # Load files
    stat = np.load(suite2p_path / 'stat.npy', allow_pickle=True)
    iscell = np.load(suite2p_path / 'iscell.npy')
    ops = np.load(suite2p_path / 'ops.npy', allow_pickle=True).item()
    F = np.load(suite2p_path / 'F.npy')
    
    # Get mean image for visualization
    if 'meanImg' in ops:
        mean_img = ops['meanImg']
    elif 'meanImg_chan2' in ops:
        mean_img = ops['meanImg_chan2']
    else:
        print("Warning: No mean image found in ops, will create blank background")
        mean_img = np.zeros((ops['Ly'], ops['Lx']))
    
    print(f"✓ Loaded {len(stat)} total ROIs")
    print(f"✓ Fluorescence traces shape: {F.shape}")
    print(f"✓ Mean image shape: {mean_img.shape}")
    
    return {
        'stat': stat,
        'iscell': iscell,
        'ops': ops,
        'F': F,
        'mean_img': mean_img
    }


def get_good_axon_indices(iscell):
    """
    Get indices of ROIs marked as cells (iscell==1)
    
    Parameters:
    -----------
    iscell : np.ndarray
        Suite2p iscell array (n_rois, 2)
    
    Returns:
    --------
    good_indices : np.ndarray
        Indices where iscell[:,0] == 1
    """
    good_idx = np.where(iscell[:, 0] == 1)[0]
    return good_idx


# ============================================================================
# USAGE
# ============================================================================

# Specify your suite2p path
suite2p_path = r"D:\V1_SpatialModulation\2p\V1_axonal\JSY060_ChronicImaging_prism\260303_JSY_JSY060_LongitudinalImaging_Axonal_Prism_Day7\TSeries-03032026-0817-001\suite2p\plane0"


# Load data
data = load_suite2p_data(suite2p_path)
good_idx = get_good_axon_indices(data['iscell'])

print(f"\n{'='*50}")
print(f"Total ROIs detected: {len(data['stat'])}")
print(f"Currently marked as good axons (iscell==1): {len(good_idx)}")
print(f"Marked as non-cells (iscell==0): {np.sum(data['iscell'][:, 0] == 0)}")
print(f"{'='*50}")


In [ ]:
# ============================================================================
# CHUNK 2: FEATURE EXTRACTION
# ============================================================================

def calculate_elongation_metrics(roi_pixels):
    """
    Calculate custom elongation metrics from ROI pixel coordinates
    
    Parameters:
    -----------
    roi_pixels : dict
        stat[i] dictionary containing 'ypix' and 'xpix'
    
    Returns:
    --------
    metrics : dict
        Dictionary with elongation, eccentricity, perimeter_area_ratio
    """
    ypix = roi_pixels['ypix']
    xpix = roi_pixels['xpix']
    
    # Create binary mask
    if len(ypix) < 3:  # Need at least 3 pixels
        return {
            'elongation': np.nan,
            'eccentricity': np.nan,
            'perimeter_area_ratio': np.nan,
            'major_axis_length': np.nan,
            'minor_axis_length': np.nan
        }
    
    # Create small bounding box around ROI
    y_min, y_max = ypix.min(), ypix.max()
    x_min, x_max = xpix.min(), xpix.max()
    
    # Create binary mask
    mask = np.zeros((y_max - y_min + 3, x_max - x_min + 3), dtype=bool)
    mask[ypix - y_min + 1, xpix - x_min + 1] = True
    
    # Use regionprops to get shape metrics
    props = regionprops(mask.astype(int))[0]
    
    # Use new API names (axis_minor_length, axis_major_length)
    minor_axis = props.axis_minor_length
    major_axis = props.axis_major_length
    
    # Calculate elongation (0 = circular, approaching 1 = very elongated)
    if minor_axis > 0:
        elongation = 1 - (minor_axis / major_axis)
    else:
        elongation = np.nan
    
    # Perimeter^2 / Area ratio (higher for elongated structures)
    perimeter_area_ratio = (props.perimeter ** 2) / (4 * np.pi * props.area)
    
    return {
        'elongation': elongation,
        'eccentricity': props.eccentricity,
        'perimeter_area_ratio': perimeter_area_ratio,
        'major_axis_length': major_axis,
        'minor_axis_length': minor_axis
    }


def calculate_trace_metrics(F_trace, frame_rate=10.047):
    """
    Calculate signal quality metrics from calcium trace.
    Kurtosis is computed on dFF (Fisher/excess kurtosis).
    Inactive traces (flat noise) have kurtosis near 0; traces with
    genuine calcium transients have kurtosis >> 0.
    
    Parameters:
    -----------
    F_trace : np.ndarray
        Fluorescence trace (1D array)
    frame_rate : float
        Imaging frame rate in Hz
    
    Returns:
    --------
    metrics : dict
        Dictionary with kurtosis, baseline_std, mean_amplitude, etc.
    """
    # Baseline estimation (bottom 20th percentile)
    baseline = np.percentile(F_trace, 20)
    baseline_std = np.std(F_trace[F_trace < np.percentile(F_trace, 50)])
    
    # Signal amplitude
    mean_amplitude = np.mean(F_trace)
    peak_amplitude = np.percentile(F_trace, 99)
    
    # dFF (delta F/F) using baseline as F0
    if baseline > 0:
        dFF = (F_trace - baseline) / baseline
    else:
        dFF = F_trace - baseline  # fallback if baseline <= 0
    
    # Fisher kurtosis of dFF (excess kurtosis; Gaussian noise = 0)
    # Active traces with sparse transients have kurtosis > 0
    dff_kurtosis = stats.kurtosis(dFF, fisher=True)
    
    # Trace variability
    cv = np.std(F_trace) / np.mean(F_trace) if np.mean(F_trace) > 0 else 0
    
    return {
        'baseline_mean': baseline,
        'baseline_std': baseline_std,
        'mean_amplitude': mean_amplitude,
        'peak_amplitude': peak_amplitude,
        'kurtosis': dff_kurtosis,
        'cv': cv
    }


def extract_all_features(data, roi_indices, frame_rate=10.047):
    """
    Extract all features for specified ROIs
    
    Parameters:
    -----------
    data : dict
        Output from load_suite2p_data()
    roi_indices : np.ndarray
        Indices of ROIs to analyze
    frame_rate : float
        Imaging frame rate in Hz
    
    Returns:
    --------
    features_df : pd.DataFrame
        DataFrame with all features for each ROI
    """
    stat = data['stat']
    F = data['F']
    
    features_list = []
    
    print(f"Extracting features for {len(roi_indices)} ROIs...")
    
    for idx in roi_indices:
        roi_stat = stat[idx]
        roi_trace = F[idx]
        
        # Initialize feature dictionary
        features = {'roi_index': idx}
        
        # Suite2p built-in features
        features['npix'] = roi_stat.get('npix', np.nan)
        features['compactness'] = roi_stat.get('compact', np.nan)
        features['aspect_ratio'] = roi_stat.get('aspect_ratio', np.nan)
        features['radius'] = roi_stat.get('radius', np.nan)
        features['med_y'] = roi_stat.get('med', [np.nan, np.nan])[0]
        features['med_x'] = roi_stat.get('med', [np.nan, np.nan])[1]
        
        # # Custom elongation metrics
        # elongation_metrics = calculate_elongation_metrics(roi_stat)
        # features.update(elongation_metrics)
        
        # Trace quality metrics
        trace_metrics = calculate_trace_metrics(roi_trace, frame_rate)
        features.update(trace_metrics)
        
        features_list.append(features)
    
    features_df = pd.DataFrame(features_list)
    
    print(f"✓ Feature extraction complete!")
    print(f"✓ Extracted {len(features_df.columns)-1} features per ROI")
    
    return features_df


# ============================================================================
# USAGE
# ============================================================================

# Extract features for all currently "good" axons
features_df = extract_all_features(data, good_idx, frame_rate=10.047)  # Update frame_rate if needed

# Display summary statistics
print("\n" + "="*70)
print("FEATURE SUMMARY STATISTICS")
print("="*70)
print(features_df.describe())

# Show first few ROIs
print("\n" + "="*70)
print("FIRST 5 ROIs:")
print("="*70)
print(features_df.head())


In [ ]:
# ============================================================================
# CHUNK 3: INTERACTIVE VISUALIZATION
# ============================================================================

def create_roi_mask(stat_entry, img_shape):
    """
    Create a binary mask for an ROI
    
    Parameters:
    -----------
    stat_entry : dict
        Single entry from stat array
    img_shape : tuple
        (height, width) of the image
    
    Returns:
    --------
    mask : np.ndarray
        Binary mask of the ROI
    """
    mask = np.zeros(img_shape, dtype=bool)
    ypix = stat_entry['ypix']
    xpix = stat_entry['xpix']
    mask[ypix, xpix] = True
    return mask


def get_roi_perimeter(mask):
    """
    Get perimeter coordinates of an ROI mask
    
    Parameters:
    -----------
    mask : np.ndarray
        Binary mask of ROI
    
    Returns:
    --------
    perimeter_coords : tuple
        (y_coords, x_coords) of perimeter pixels
    """
    # Dilate mask slightly and subtract to get perimeter
    from scipy.ndimage import binary_dilation
    dilated = binary_dilation(mask, iterations=1)
    perimeter = dilated & ~mask
    return np.where(perimeter)


def calculate_suspiciousness_score(row):
    """
    Calculate a "suspiciousness" score for each ROI
    Lower score = LESS suspicious (better axon candidate)
    
    Good axons should have:
    - High elongation
    - High dFF kurtosis (sparse, large transients)
    - Low compactness
    - Reasonable size (not too small or huge)
    
    Parameters:
    -----------
    row : pd.Series
        Single row from features_df
    
    Returns:
    --------
    score : float
        Suspiciousness score (lower = better)
    """
    score = 0
    
    # # Penalize low elongation (good axons are elongated)
    # if row['elongation'] < 0.3:
    #     score += (0.3 - row['elongation']) * 10
    
    # # Penalize high compactness (circular = suspicious)
    # if row['compactness'] > 1.2:
    #     score += (row['compactness'] - 1.2) * 5
    
    # Penalize low dFF kurtosis (flat/noisy traces)
    if row['kurtosis'] < 2:
        score += (2 - row['kurtosis']) * 2
    
    # Penalize very small ROIs
    if row['npix'] < 40:
        score += (40 - row['npix']) * 0.1
    
    # Penalize very large ROIs (might be merged)
    if row['npix'] > 300:
        score += (row['npix'] - 300) * 0.01
    
    return score


def visualize_roi_interactive(data, features_df, sort_by_suspiciousness=True):
    """
    Create interactive viewer to browse through ROIs
    Shows ROI perimeter + calcium trace
    
    Parameters:
    -----------
    data : dict
        Output from load_suite2p_data()
    features_df : pd.DataFrame
        Features DataFrame
    sort_by_suspiciousness : bool
        If True, sort by suspiciousness (least suspicious first)
    """
    # Calculate suspiciousness scores
    features_df['suspiciousness'] = features_df.apply(calculate_suspiciousness_score, axis=1)
    
    # Sort by suspiciousness (least suspicious first)
    if sort_by_suspiciousness:
        sorted_df = features_df.sort_values('suspiciousness').reset_index(drop=True)
        print("Sorted by suspiciousness: LEAST suspicious (best axons) shown FIRST")
    else:
        sorted_df = features_df.copy()
    
    # Interactive widget state
    current_idx = [0]  # Use list to make it mutable in nested function
    
    # Create output widget
    output = widgets.Output()
    
    def show_roi(idx):
        """Display ROI at given index"""
        with output:
            clear_output(wait=True)
            
            row = sorted_df.iloc[idx]
            roi_idx = int(row['roi_index'])
            
            # Create figure
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            
            # ============ LEFT: ROI perimeter on mean image ============
            ax = axes[0]
            ax.imshow(data['mean_img'], cmap='gray', vmin=np.percentile(data['mean_img'], 1),
                     vmax=np.percentile(data['mean_img'], 99))
            
            # Get ROI mask and perimeter
            mask = create_roi_mask(data['stat'][roi_idx], data['mean_img'].shape)
            perim_y, perim_x = get_roi_perimeter(mask)
            
            # Plot perimeter in red
            ax.scatter(perim_x, perim_y, c='red', s=1, alpha=0.8)
            
            # Zoom to ROI region
            ypix = data['stat'][roi_idx]['ypix']
            xpix = data['stat'][roi_idx]['xpix']
            padding = 30
            ax.set_xlim(xpix.min() - padding, xpix.max() + padding)
            ax.set_ylim(ypix.max() + padding, ypix.min() - padding)  # Inverted y-axis
            
            ax.set_title(f'ROI #{roi_idx} Perimeter', fontsize=12, fontweight='bold')
            ax.axis('off')
            
            # ============ RIGHT: Calcium trace ============
            ax = axes[1]
            trace = data['F'][roi_idx]
            time = np.arange(len(trace)) / 30  # Assuming 30 Hz
            
            ax.plot(time, trace, 'k-', linewidth=0.8)
            ax.set_xlabel('Time (s)', fontsize=11)
            ax.set_ylabel('Fluorescence (a.u.)', fontsize=11)
            ax.set_title(f'Calcium Trace', fontsize=12, fontweight='bold')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            
            # ============ Add feature info ============
            info_text = (
                f"Suspiciousness: {row['suspiciousness']:.2f}\n"
                f"dFF Kurtosis: {row['kurtosis']:.2f}\n"
                f"N pixels: {int(row['npix'])}"
            )
            
            fig.text(0.02, 0.98, info_text, transform=fig.transFigure, 
                    fontsize=10, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
            
            # Add navigation info
            nav_text = f"ROI {idx + 1} / {len(sorted_df)}"
            fig.text(0.98, 0.02, nav_text, transform=fig.transFigure,
                    fontsize=10, ha='right',
                    bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
            
            plt.tight_layout()
            plt.show()
    
    # Navigation buttons
    def on_next(b):
        if current_idx[0] < len(sorted_df) - 1:
            current_idx[0] += 1
            show_roi(current_idx[0])
    
    def on_prev(b):
        if current_idx[0] > 0:
            current_idx[0] -= 1
            show_roi(current_idx[0])
    
    def on_jump(change):
        idx = change['new'] - 1  # Convert to 0-indexed
        if 0 <= idx < len(sorted_df):
            current_idx[0] = idx
            show_roi(current_idx[0])
    
    # Create buttons
    prev_button = widgets.Button(description='← Previous')
    next_button = widgets.Button(description='Next →')
    jump_box = widgets.IntText(value=1, min=1, max=len(sorted_df), 
                               description='Jump to:')
    
    prev_button.on_click(on_prev)
    next_button.on_click(on_next)
    jump_box.observe(on_jump, names='value')
    
    # Layout
    buttons = widgets.HBox([prev_button, next_button, jump_box])
    display(buttons)
    display(output)
    
    # Show first ROI
    show_roi(0)


# ============================================================================
# USAGE
# ============================================================================

# Launch interactive viewer
visualize_roi_interactive(data, features_df, sort_by_suspiciousness=True)


In [ ]:
# ============================================================================
# CHUNK 4: AUTOMATED FILTERING WITH PREVIEW
# ============================================================================

def plot_feature_distributions(features_df):
    """
    Plot distributions of key features to help set thresholds
    
    Parameters:
    -----------
    features_df : pd.DataFrame
        Features DataFrame
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes = axes.flatten()
    
    key_features = ['kurtosis', 'npix']

    for idx, feature in enumerate(key_features):
        ax = axes[idx]
        data = features_df[feature].dropna()
        
        # Histogram
        ax.hist(data, bins=100, alpha=0.7, edgecolor='black')
        
        # Add median and quartile lines
        median = data.median()
        q25 = data.quantile(0.25)
        q75 = data.quantile(0.75)
        
        ax.axvline(median, color='red', linestyle='--', linewidth=2, label=f'Median: {median:.2f}')
        ax.axvline(q25, color='orange', linestyle=':', linewidth=1.5, label=f'Q25: {q25:.2f}')
        ax.axvline(q75, color='orange', linestyle=':', linewidth=1.5, label=f'Q75: {q75:.2f}')
        
        ax.set_xlabel(feature, fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title(f'{feature} Distribution', fontsize=12, fontweight='bold')
        ax.legend(fontsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*70)
    print("KEY FEATURE STATISTICS (for threshold setting)")
    print("="*70)
    for feature in key_features:
        data = features_df[feature].dropna()
        print(f"\n{feature}:")
        print(f"  5th percentile:  {data.quantile(0.05):.3f}")
        print(f"  25th percentile: {data.quantile(0.25):.3f}")
        print(f"  Median:          {data.quantile(0.50):.3f}")
        print(f"  75th percentile: {data.quantile(0.75):.3f}")
        print(f"  95th percentile: {data.quantile(0.95):.3f}")


def apply_axon_filters(features_df, thresholds):
    """
    Apply threshold filters to identify good axons
    
    Parameters:
    -----------
    features_df : pd.DataFrame
        Features DataFrame
    thresholds : dict
        Dictionary of feature thresholds
        Example: {
            'kurtosis_min': 1.0,
            'npix_min': 5,
            'npix_max': 250
        }
    
    Returns:
    --------
    keep_mask : np.ndarray (bool)
        Boolean mask indicating which ROIs to keep
    rejection_reasons : dict
        Dictionary mapping roi_index to list of rejection reasons
    """
    keep_mask = np.ones(len(features_df), dtype=bool)
    rejection_reasons = {}
    
    for idx, row in features_df.iterrows():
        reasons = []
        
        # Check each threshold
        # if 'elongation_min' in thresholds and row['elongation'] < thresholds['elongation_min']:
        #     reasons.append(f"Low elongation ({row['elongation']:.3f} < {thresholds['elongation_min']})")
        
        # if 'compactness_max' in thresholds and row['compactness'] > thresholds['compactness_max']:
        #     reasons.append(f"Too compact ({row['compactness']:.3f} > {thresholds['compactness_max']})")
        
        if 'kurtosis_min' in thresholds and row['kurtosis'] < thresholds['kurtosis_min']:
            reasons.append(f"Low dFF kurtosis ({row['kurtosis']:.2f} < {thresholds['kurtosis_min']})")
        
        if 'npix_min' in thresholds and row['npix'] < thresholds['npix_min']:
            reasons.append(f"Too small ({int(row['npix'])} < {thresholds['npix_min']})")
        
        if 'npix_max' in thresholds and row['npix'] > thresholds['npix_max']:
            reasons.append(f"Too large ({int(row['npix'])} > {thresholds['npix_max']})")
        
        # if 'perimeter_area_ratio_max' in thresholds and row['perimeter_area_ratio'] > thresholds['perimeter_area_ratio_max']:
        #     reasons.append(f"High perim/area ({row['perimeter_area_ratio']:.2f} > {thresholds['perimeter_area_ratio_max']})")
        
        # if 'aspect_ratio_min' in thresholds and row['aspect_ratio'] < thresholds['aspect_ratio_min']:
        #     reasons.append(f"Low aspect ratio ({row['aspect_ratio']:.3f} < {thresholds['aspect_ratio_min']})")
        
        # If any reasons exist, mark for removal
        if reasons:
            keep_mask[idx] = False
            rejection_reasons[int(row['roi_index'])] = reasons
    
    return keep_mask, rejection_reasons


def preview_rejections(data, features_df, keep_mask, rejection_reasons, n_preview=12):
    """
    Preview a random sample of ROIs that will be rejected
    
    Parameters:
    -----------
    data : dict
        Suite2p data
    features_df : pd.DataFrame
        Features DataFrame
    keep_mask : np.ndarray
        Boolean mask of ROIs to keep
    rejection_reasons : dict
        Rejection reasons for each ROI
    n_preview : int
        Number of rejections to preview
    """
    reject_df = features_df[~keep_mask]
    
    if len(reject_df) == 0:
        print("No ROIs would be rejected with these thresholds!")
        return
    
    # Sample random rejections
    n_show = min(n_preview, len(reject_df))
    sample_df = reject_df.sample(n=n_show)
    
    # Create grid
    n_cols = 4
    n_rows = int(np.ceil(n_show / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(sample_df.iterrows()):
        ax = axes[idx]
        roi_idx = int(row['roi_index'])
        
        # Show ROI perimeter
        ax.imshow(data['mean_img'], cmap='gray', 
                 vmin=np.percentile(data['mean_img'], 1),
                 vmax=np.percentile(data['mean_img'], 99))
        
        mask = create_roi_mask(data['stat'][roi_idx], data['mean_img'].shape)
        perim_y, perim_x = get_roi_perimeter(mask)
        ax.scatter(perim_x, perim_y, c='red', s=1, alpha=0.8)
        
        # Zoom to ROI
        ypix = data['stat'][roi_idx]['ypix']
        xpix = data['stat'][roi_idx]['xpix']
        padding = 20
        ax.set_xlim(xpix.min() - padding, xpix.max() + padding)
        ax.set_ylim(ypix.max() + padding, ypix.min() - padding)
        
        # Title with rejection reasons
        reasons_text = '\n'.join(rejection_reasons[roi_idx])
        ax.set_title(f'ROI #{roi_idx}\n{reasons_text}', fontsize=7)
        ax.axis('off')
    
    # Hide unused subplots
    for idx in range(n_show, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print(f"\nShowing {n_show} random examples of {len(reject_df)} total rejections")


def preview_kept_rois(data, features_df, keep_mask, n_preview=12):
    """
    Preview a random sample of ROIs that will be KEPT
    
    Parameters:
    -----------
    data : dict
        Suite2p data
    features_df : pd.DataFrame
        Features DataFrame
    keep_mask : np.ndarray
        Boolean mask of ROIs to keep
    n_preview : int
        Number of kept ROIs to preview
    """
    keep_df = features_df[keep_mask]
    
    if len(keep_df) == 0:
        print("No ROIs would be kept with these thresholds!")
        return
    
    # Sample random kept ROIs
    n_show = min(n_preview, len(keep_df))
    sample_df = keep_df.sample(n=n_show)
    
    # Create grid
    n_cols = 4
    n_rows = int(np.ceil(n_show / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(sample_df.iterrows()):
        ax = axes[idx]
        roi_idx = int(row['roi_index'])
        
        # Show ROI perimeter
        ax.imshow(data['mean_img'], cmap='gray', 
                 vmin=np.percentile(data['mean_img'], 1),
                 vmax=np.percentile(data['mean_img'], 99))
        
        mask = create_roi_mask(data['stat'][roi_idx], data['mean_img'].shape)
        perim_y, perim_x = get_roi_perimeter(mask)
        ax.scatter(perim_x, perim_y, c='lime', s=1, alpha=0.8)  # Green for kept
        
        # Zoom to ROI
        ypix = data['stat'][roi_idx]['ypix']
        xpix = data['stat'][roi_idx]['xpix']
        padding = 20
        ax.set_xlim(xpix.min() - padding, xpix.max() + padding)
        ax.set_ylim(ypix.max() + padding, ypix.min() - padding)
        
        # Title with key features
        info_text = (f"Kurtosis: {row['kurtosis']:.2f}")
        ax.set_title(f'ROI #{roi_idx} ✓\n{info_text}', fontsize=8, color='green')
        ax.axis('off')
    
    # Hide unused subplots
    for idx in range(n_show, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print(f"\nShowing {n_show} random examples of {len(keep_df)} total KEPT ROIs")


def print_filter_summary(features_df, keep_mask, rejection_reasons):
    """
    Print summary statistics about filtering results
    """
    n_total = len(features_df)
    n_keep = keep_mask.sum()
    n_reject = (~keep_mask).sum()
    
    print("\n" + "="*70)
    print("FILTERING SUMMARY")
    print("="*70)
    print(f"Total ROIs:              {n_total}")
    print(f"ROIs to KEEP:            {n_keep} ({100*n_keep/n_total:.1f}%)")
    print(f"ROIs to REJECT:          {n_reject} ({100*n_reject/n_total:.1f}%)")
    
    # Count rejection reasons
    if rejection_reasons:
        print(f"\n{'Rejection Reason':<50} {'Count':<10}")
        print("-"*70)
        
        reason_counts = {}
        for reasons_list in rejection_reasons.values():
            for reason in reasons_list:
                key = reason.split('(')[0].strip()  # Get just the reason type
                reason_counts[key] = reason_counts.get(key, 0) + 1
        
        for reason, count in sorted(reason_counts.items(), key=lambda x: -x[1]):
            print(f"{reason:<50} {count:<10}")


# ============================================================================
# USAGE WORKFLOW
# ============================================================================

print("STEP 1: Visualize feature distributions to set thresholds")
print("="*70)
plot_feature_distributions(features_df)

# STEP 2: Set your thresholds based on the distributions above
print("\n\nSTEP 2: Define filtering thresholds")
print("="*70)
print("Review the distributions above and set thresholds below...")

thresholds = {
    'kurtosis_min': 1.0,           # Discard flat/noisy traces (dFF Fisher kurtosis < 1.0)
    'npix_min': 5,                 # Not too small
    'npix_max': 250,               # Not too large (might be merged)
}

print("Current thresholds:")
for key, value in thresholds.items():
    print(f"  {key}: {value}")

# STEP 3: Apply filters and preview
print("\n\nSTEP 3: Apply filters and preview results")
print("="*70)
keep_mask, rejection_reasons = apply_axon_filters(features_df, thresholds)
print_filter_summary(features_df, keep_mask, rejection_reasons)

# STEP 4a: Preview random sample of KEPT ROIs (in GREEN)
print("\n\nSTEP 4a: Preview random sample of KEPT ROIs")
print("="*70)
preview_kept_rois(data, features_df, keep_mask, n_preview=12)

# STEP 4b: Preview random sample of REJECTED ROIs (in RED)
print("\n\nSTEP 4b: Preview random sample of REJECTED ROIs")
print("="*70)
preview_rejections(data, features_df, keep_mask, rejection_reasons, n_preview=12)


In [ ]:
# ============================================================================
# DIAGNOSTIC: Kurtosis-rejected cells — what do they look like?
# ============================================================================
# Fisher excess kurtosis of dFF:
#   ~0       → Gaussian-like noise, no transients (flat/noisy trace)
#   negative → Platykurtic; e.g. persistently elevated (always-on)
#   > 1      → Sparse, large transients (good axon signal)
#
# What to look for in rejected cells:
#   "Flat noise"    : trace fluctuates around baseline with no clear events
#   "Always active" : trace stays persistently elevated; sparse structure lost
#   "Weak signal"   : some peaks but too small/infrequent to pass threshold
# ============================================================================

def categorize_kurtosis_trace(F_trace, frame_rate=10.047):
    """
    Categorize a low-kurtosis trace into its likely failure mode.
    Returns: 'flat_noise', 'always_active', or 'weak_signal'
    """
    baseline = np.percentile(F_trace, 20)
    dFF = (F_trace - baseline) / baseline if baseline > 0 else F_trace - baseline

    mean_dff   = np.mean(dFF)
    pct10_dff  = np.percentile(dFF, 10)   # if even bottom 10% is elevated → always active

    if mean_dff > 0.3 and pct10_dff > 0.1:
        return 'always_active'

    cv = np.std(F_trace) / np.mean(F_trace) if np.mean(F_trace) > 0 else 0
    peak_to_noise = (np.percentile(F_trace, 99) - baseline) / (np.std(F_trace) + 1e-9)
    if peak_to_noise < 2.0 or cv < 0.05:
        return 'flat_noise'

    return 'weak_signal'


def diagnose_kurtosis_rejections(data, features_df, keep_mask, rejection_reasons,
                                  n_preview=10, frame_rate=10.047):
    """
    Show kurtosis-rejected ROIs sorted from worst (lowest) kurtosis upward.
    Each row: left = ROI morphology, right = dFF trace.
    
    Helps calibrate the kurtosis threshold and understand what is being removed.
    
    Parameters
    ----------
    n_preview : int   — how many examples to show (worst kurtosis first)
    """
    from collections import Counter

    reject_df = features_df[~keep_mask].copy()

    # Keep only the kurtosis-rejected subset
    kurt_rejected = reject_df[
        reject_df['roi_index'].apply(
            lambda ri: any('kurtosis' in r.lower()
                           for r in rejection_reasons.get(int(ri), []))
        )
    ].sort_values('kurtosis').reset_index(drop=True)

    if len(kurt_rejected) == 0:
        print("No kurtosis-rejected ROIs found.")
        return

    n_show = min(n_preview, len(kurt_rejected))
    sample_df = kurt_rejected.head(n_show)

    # Category breakdown across ALL kurtosis-rejected cells
    all_cats = [categorize_kurtosis_trace(data['F'][int(r['roi_index'])], frame_rate)
                for _, r in kurt_rejected.iterrows()]

    cat_colors = {'flat_noise': 'steelblue', 'always_active': 'darkorange', 'weak_signal': 'mediumpurple'}
    cat_labels = {'flat_noise': 'Flat noise', 'always_active': 'Always active', 'weak_signal': 'Weak signal'}

    print(f"Kurtosis-rejected ROIs: {len(kurt_rejected)} total  |  showing worst {n_show}")
    print(f"\nCategory breakdown (all {len(kurt_rejected)} kurtosis rejects):")
    for cat, count in Counter(all_cats).most_common():
        print(f"  {cat_labels[cat]:<20}: {count}  ({100*count/len(kurt_rejected):.0f}%)")
    print()

    # Plot
    fig, axes = plt.subplots(n_show, 2, figsize=(14, 3 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for i, (_, row) in enumerate(sample_df.iterrows()):
        roi_idx = int(row['roi_index'])
        cat     = categorize_kurtosis_trace(data['F'][roi_idx], frame_rate)
        color   = cat_colors[cat]

        # --- LEFT: morphology ---
        ax_img = axes[i, 0]
        ax_img.imshow(data['mean_img'], cmap='gray',
                      vmin=np.percentile(data['mean_img'], 1),
                      vmax=np.percentile(data['mean_img'], 99))

        mask   = create_roi_mask(data['stat'][roi_idx], data['mean_img'].shape)
        py, px = get_roi_perimeter(mask)
        ax_img.scatter(px, py, c='red', s=1, alpha=0.9)

        ypix = data['stat'][roi_idx]['ypix']
        xpix = data['stat'][roi_idx]['xpix']
        pad  = 25
        ax_img.set_xlim(xpix.min() - pad, xpix.max() + pad)
        ax_img.set_ylim(ypix.max() + pad, ypix.min() - pad)
        ax_img.axis('off')
        ax_img.set_title(f'ROI #{roi_idx}  |  Kurt={row["kurtosis"]:.2f}  |  {cat_labels[cat]}',
                         fontsize=9, color=color, fontweight='bold')

        # --- RIGHT: dFF trace ---
        ax_tr = axes[i, 1]
        F_trace  = data['F'][roi_idx]
        baseline = np.percentile(F_trace, 20)
        dFF      = (F_trace - baseline) / baseline if baseline > 0 else F_trace - baseline
        t        = np.arange(len(dFF)) / frame_rate

        ax_tr.plot(t, dFF, color=color, linewidth=0.7)
        ax_tr.axhline(0, color='k', linewidth=0.5, linestyle='--', alpha=0.4)
        ax_tr.set_xlabel('Time (s)', fontsize=8)
        ax_tr.set_ylabel('dF/F', fontsize=8)
        ax_tr.spines['top'].set_visible(False)
        ax_tr.spines['right'].set_visible(False)
        ax_tr.tick_params(labelsize=7)
        ax_tr.set_title(f'Kurt={row["kurtosis"]:.2f}  |  CV={row["cv"]:.2f}  |  npix={int(row["npix"])}',
                        fontsize=8)

    plt.suptitle(
        f'Kurtosis-rejected ROIs  —  worst {n_show} shown (sorted low → high kurtosis)\n'
        'Blue = flat noise  |  Orange = always active  |  Purple = weak signal',
        fontsize=11, fontweight='bold', y=1.005
    )
    plt.tight_layout()
    plt.show()

    return kurt_rejected


# ============================================================================
# RUN — requires keep_mask and rejection_reasons from Chunk 4
# ============================================================================

kurt_rejected_df = diagnose_kurtosis_rejections(
    data, features_df, keep_mask, rejection_reasons,
    n_preview=10,       # adjust: how many examples to inspect
    frame_rate=10.047
)

In [ ]:
# ============================================================================
# DIAGNOSTIC: Cells that PASSED all filters (size + kurtosis)
# ============================================================================

def show_passed_rois(data, features_df, keep_mask, n_preview=10, frame_rate=10.047,
                     sort_by='kurtosis', ascending=True):
    """
    Show ROIs that passed all filters, with morphology + dFF trace side-by-side.
    Sorted by kurtosis (lowest-passing first by default) so you can see borderline cases.

    Parameters
    ----------
    sort_by   : column to sort by (default 'kurtosis'; ascending=True shows borderline first)
    """
    keep_df = features_df[keep_mask].copy().sort_values(sort_by, ascending=ascending).reset_index(drop=True)

    if len(keep_df) == 0:
        print("No ROIs passed the filters.")
        return

    n_show   = min(n_preview, len(keep_df))
    sample_df = keep_df.head(n_show)

    print(f"Passed ROIs: {len(keep_df)} total  |  showing {n_show} (sorted by {sort_by}, ascending={ascending})")
    print(f"  kurtosis range in passed set: {keep_df['kurtosis'].min():.2f} – {keep_df['kurtosis'].max():.2f}")
    print(f"  npix range:                   {int(keep_df['npix'].min())} – {int(keep_df['npix'].max())}")
    print()

    fig, axes = plt.subplots(n_show, 2, figsize=(14, 3 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]

    for i, (_, row) in enumerate(sample_df.iterrows()):
        roi_idx = int(row['roi_index'])

        # --- LEFT: morphology ---
        ax_img = axes[i, 0]
        ax_img.imshow(data['mean_img'], cmap='gray',
                      vmin=np.percentile(data['mean_img'], 1),
                      vmax=np.percentile(data['mean_img'], 99))

        mask   = create_roi_mask(data['stat'][roi_idx], data['mean_img'].shape)
        py, px = get_roi_perimeter(mask)
        ax_img.scatter(px, py, c='lime', s=1, alpha=0.9)

        ypix = data['stat'][roi_idx]['ypix']
        xpix = data['stat'][roi_idx]['xpix']
        pad  = 25
        ax_img.set_xlim(xpix.min() - pad, xpix.max() + pad)
        ax_img.set_ylim(ypix.max() + pad, ypix.min() - pad)
        ax_img.axis('off')
        ax_img.set_title(f'ROI #{roi_idx}  |  Kurt={row["kurtosis"]:.2f}  |  npix={int(row["npix"])}',
                         fontsize=9, color='green', fontweight='bold')

        # --- RIGHT: dFF trace ---
        ax_tr = axes[i, 1]
        F_trace  = data['F'][roi_idx]
        baseline = np.percentile(F_trace, 20)
        dFF      = (F_trace - baseline) / baseline if baseline > 0 else F_trace - baseline
        t        = np.arange(len(dFF)) / frame_rate

        ax_tr.plot(t, dFF, color='green', linewidth=0.7)
        ax_tr.axhline(0, color='k', linewidth=0.5, linestyle='--', alpha=0.4)
        ax_tr.set_xlabel('Time (s)', fontsize=8)
        ax_tr.set_ylabel('dF/F', fontsize=8)
        ax_tr.spines['top'].set_visible(False)
        ax_tr.spines['right'].set_visible(False)
        ax_tr.tick_params(labelsize=7)
        ax_tr.set_title(f'Kurt={row["kurtosis"]:.2f}  |  CV={row["cv"]:.2f}  |  npix={int(row["npix"])}',
                        fontsize=8)

    plt.suptitle(
        f'Passed ROIs (size + kurtosis)  —  {n_show} shown, sorted by {sort_by} ascending={ascending}\n'
        'Green outline = kept',
        fontsize=11, fontweight='bold', y=1.005
    )
    plt.tight_layout()
    plt.show()


# ============================================================================
# RUN — requires keep_mask from Chunk 4
# ============================================================================

# sort_by='kurtosis', ascending=True  → shows borderline passes first (lowest kurtosis that still passed)
# sort_by='kurtosis', ascending=False → shows the best-quality cells first
show_passed_rois(
    data, features_df, keep_mask,
    n_preview=50,
    frame_rate=10.047,
    sort_by='kurtosis',
    ascending=True      # True = borderline first; False = best first
)

In [ ]:
# ============================================================================
# CHUNK 5: FINAL CONFIRMATION AND SAVING
# ============================================================================

def plot_all_rois_overlay(data, features_df, keep_mask):
    """
    Create two side-by-side figures:
    1. Mean image with ALL current ROIs (what you have now)
    2. Mean image with only KEPT ROIs (what you'll have after filtering)
    
    Parameters:
    -----------
    data : dict
        Suite2p data
    features_df : pd.DataFrame
        Features DataFrame
    keep_mask : np.ndarray
        Boolean mask of ROIs to keep
    """
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    # ============ LEFT: All current ROIs ============
    ax = axes[0]
    ax.imshow(data['mean_img'], cmap='gray', 
             vmin=np.percentile(data['mean_img'], 1),
             vmax=np.percentile(data['mean_img'], 99))
    
    # Draw all current ROIs in cyan
    for _, row in features_df.iterrows():
        roi_idx = int(row['roi_index'])
        mask = create_roi_mask(data['stat'][roi_idx], data['mean_img'].shape)
        perim_y, perim_x = get_roi_perimeter(mask)
        ax.scatter(perim_x, perim_y, c='cyan', s=0.5, alpha=0.6)
    
    ax.set_title(f'BEFORE Filtering: All {len(features_df)} ROIs', 
                fontsize=14, fontweight='bold')
    ax.axis('off')
    
    # ============ RIGHT: Only kept ROIs ============
    ax = axes[1]
    ax.imshow(data['mean_img'], cmap='gray', 
             vmin=np.percentile(data['mean_img'], 1),
             vmax=np.percentile(data['mean_img'], 99))
    
    # Draw only kept ROIs in green
    keep_df = features_df[keep_mask]
    for _, row in keep_df.iterrows():
        roi_idx = int(row['roi_index'])
        mask = create_roi_mask(data['stat'][roi_idx], data['mean_img'].shape)
        perim_y, perim_x = get_roi_perimeter(mask)
        ax.scatter(perim_x, perim_y, c='lime', s=0.5, alpha=0.6)
    
    ax.set_title(f'AFTER Filtering: {len(keep_df)} Kept ROIs', 
                fontsize=14, fontweight='bold', color='green')
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()


def plot_rejected_rois_overlay(data, features_df, keep_mask):
    """
    Show mean image with ONLY rejected ROIs highlighted in red
    
    Parameters:
    -----------
    data : dict
        Suite2p data
    features_df : pd.DataFrame
        Features DataFrame
    keep_mask : np.ndarray
        Boolean mask of ROIs to keep
    """
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    
    ax.imshow(data['mean_img'], cmap='gray', 
             vmin=np.percentile(data['mean_img'], 1),
             vmax=np.percentile(data['mean_img'], 99))
    
    # Draw only rejected ROIs in red
    reject_df = features_df[~keep_mask]
    for _, row in reject_df.iterrows():
        roi_idx = int(row['roi_index'])
        mask = create_roi_mask(data['stat'][roi_idx], data['mean_img'].shape)
        perim_y, perim_x = get_roi_perimeter(mask)
        ax.scatter(perim_x, perim_y, c='red', s=0.5, alpha=0.7)
    
    ax.set_title(f'Rejected ROIs Only: {len(reject_df)} ROIs to be removed', 
                fontsize=14, fontweight='bold', color='red')
    ax.axis('off')
    
    plt.tight_layout()
    plt.show()


def save_updated_iscell(data, features_df, keep_mask, suite2p_path, backup=True):
    """
    Update and save the iscell.npy file
    
    Parameters:
    -----------
    data : dict
        Suite2p data
    features_df : pd.DataFrame
        Features DataFrame
    keep_mask : np.ndarray
        Boolean mask of ROIs to keep
    suite2p_path : str or Path
        Path to suite2p/plane0 folder
    backup : bool
        If True, create backup of original iscell.npy
    """
    suite2p_path = Path(suite2p_path)
    
    # Create a copy of iscell array
    iscell_updated = data['iscell'].copy()
    
    # Get indices of ROIs to reject (from features_df)
    reject_indices = features_df[~keep_mask]['roi_index'].values.astype(int)
    
    # Set rejected ROIs to 0
    iscell_updated[reject_indices, 0] = 0
    
    # Create backup if requested
    if backup:
        backup_path = suite2p_path / 'iscell_backup.npy'
        np.save(backup_path, data['iscell'])
        print(f"✓ Backup saved to: {backup_path}")
    
    # Save updated iscell
    iscell_path = suite2p_path / 'iscell.npy'
    np.save(iscell_path, iscell_updated)
    print(f"✓ Updated iscell.npy saved to: {iscell_path}")
    
    # Print summary
    n_original_cells = np.sum(data['iscell'][:, 0] == 1)
    n_updated_cells = np.sum(iscell_updated[:, 0] == 1)
    
    print(f"\n{'='*70}")
    print("FINAL RESULTS")
    print(f"{'='*70}")
    print(f"Original good cells (iscell==1):  {n_original_cells}")
    print(f"Updated good cells (iscell==1):   {n_updated_cells}")
    print(f"ROIs removed:                     {n_original_cells - n_updated_cells}")
    print(f"{'='*70}")


# ============================================================================
# USAGE: FINAL CONFIRMATION WORKFLOW
# ============================================================================

print("="*70)
print("FINAL CONFIRMATION BEFORE SAVING")
print("="*70)

# Show summary one more time
print_filter_summary(features_df, keep_mask, rejection_reasons)

# Figure 1: Side-by-side comparison (Before vs After)
print("\n\nFigure 1: Before vs After Filtering")
print("="*70)
plot_all_rois_overlay(data, features_df, keep_mask)

# Figure 2: Show only rejected ROIs
print("\n\nFigure 2: Rejected ROIs Only (in RED)")
print("="*70)
plot_rejected_rois_overlay(data, features_df, keep_mask)


In [ ]:
# ============================================================================
# DIAGNOSTIC: Why are elongated ROIs being rejected?
# ============================================================================

def diagnose_elongated_rejections(features_df, keep_mask, rejection_reasons, elongation_threshold=0.4):
    """
    Find elongated ROIs that are being rejected and figure out why
    
    Parameters:
    -----------
    features_df : pd.DataFrame
        Features DataFrame
    keep_mask : np.ndarray
        Boolean mask of ROIs to keep
    rejection_reasons : dict
        Rejection reasons
    elongation_threshold : float
        Minimum elongation to be considered "elongated"
    """
    # Find rejected ROIs that are elongated
    reject_df = features_df[~keep_mask]
    elongated_rejects = reject_df[reject_df['elongation'] > elongation_threshold]
    
    if len(elongated_rejects) == 0:
        print(f"No elongated ROIs (elongation > {elongation_threshold}) are being rejected.")
        return
    
    print(f"\n{'='*70}")
    print(f"DIAGNOSTIC: {len(elongated_rejects)} ELONGATED ROIs are being REJECTED")
    print(f"{'='*70}")
    
    # Count rejection reasons for elongated ROIs
    reason_counts = {}
    for roi_idx in elongated_rejects['roi_index'].values:
        reasons = rejection_reasons[int(roi_idx)]
        for reason in reasons:
            key = reason.split('(')[0].strip()
            reason_counts[key] = reason_counts.get(key, 0) + 1
    
    print(f"\nWhy are elongated axons being rejected?")
    print(f"{'Reason':<40} {'Count':<10} {'% of elongated rejects':<20}")
    print("-"*70)
    for reason, count in sorted(reason_counts.items(), key=lambda x: -x[1]):
        pct = 100 * count / len(elongated_rejects)
        print(f"{reason:<40} {count:<10} {pct:.1f}%")
    
    # Show distribution of features for elongated rejects
    print(f"\n{'='*70}")
    print("Feature distributions for elongated rejected ROIs:")
    print(f"{'='*70}")
    
    features_to_check = ['elongation', 'compactness', 'snr', 'npix', 'aspect_ratio', 'perimeter_area_ratio']
    
    for feature in features_to_check:
        vals = elongated_rejects[feature].dropna()
        if len(vals) > 0:
            print(f"\n{feature}:")
            print(f"  Min:    {vals.min():.3f}")
            print(f"  25th:   {vals.quantile(0.25):.3f}")
            print(f"  Median: {vals.median():.3f}")
            print(f"  75th:   {vals.quantile(0.75):.3f}")
            print(f"  Max:    {vals.max():.3f}")
    
    return elongated_rejects


def compare_kept_vs_rejected_elongated(features_df, keep_mask):
    """
    Compare feature distributions between kept and rejected elongated ROIs
    """
    keep_df = features_df[keep_mask]
    reject_df = features_df[~keep_mask]
    
    # Focus on elongated ROIs
    elongated_kept = keep_df[keep_df['elongation'] > 0.4]
    elongated_rejected = reject_df[reject_df['elongation'] > 0.4]
    
    if len(elongated_rejected) == 0:
        print("No elongated ROIs are being rejected!")
        return
    
    # Plot comparison
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    features_to_compare = ['elongation', 'compactness', 'snr', 'npix', 'aspect_ratio', 'perimeter_area_ratio']
    
    for idx, feature in enumerate(features_to_compare):
        ax = axes[idx]
        
        # Plot histograms
        if len(elongated_kept) > 0:
            ax.hist(elongated_kept[feature].dropna(), bins=30, alpha=0.5, 
                   label=f'Kept (n={len(elongated_kept)})', color='green', edgecolor='black')
        
        if len(elongated_rejected) > 0:
            ax.hist(elongated_rejected[feature].dropna(), bins=30, alpha=0.5, 
                   label=f'Rejected (n={len(elongated_rejected)})', color='red', edgecolor='black')
        
        ax.set_xlabel(feature, fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title(f'{feature}\n(Elongated ROIs only)', fontsize=11, fontweight='bold')
        ax.legend()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    print("\nGreen = Kept elongated ROIs")
    print("Red = Rejected elongated ROIs")
    print("\nLook for features where red and green don't overlap much - those are causing the rejections!")


# # ============================================================================
# # RUN DIAGNOSTICS
# # ============================================================================

# print("Running diagnostics on rejected elongated ROIs...")
# print("="*70)

# # Find elongated ROIs that are being rejected
# elongated_rejects = diagnose_elongated_rejections(features_df, keep_mask, rejection_reasons, 
#                                                    elongation_threshold=0.4)

# # Compare distributions
# print("\n\nComparing feature distributions: Kept vs Rejected (elongated ROIs only)")
# print("="*70)
# compare_kept_vs_rejected_elongated(features_df, keep_mask)

# # Show some examples of rejected elongated ROIs
# if elongated_rejects is not None and len(elongated_rejects) > 0:
#     print("\n\nShowing examples of REJECTED elongated ROIs:")
#     print("="*70)
    
#     # Create temporary mask for just these elongated rejects
#     temp_keep_mask = np.ones(len(features_df), dtype=bool)
#     temp_keep_mask[elongated_rejects.index] = False
    
#     # Show them using the preview function
#     preview_rejections(data, features_df, temp_keep_mask, rejection_reasons, n_preview=12)

In [ ]:
# ============================================================================
# SAVE UPDATED ISCELL WITH ORIGINAL BACKUP
# ============================================================================

def save_iscell_with_backup(data, features_df, keep_mask, suite2p_path):
    """
    Save updated iscell.npy and rename original to iscell_og.npy
    
    Parameters:
    -----------
    data : dict
        Suite2p data
    features_df : pd.DataFrame
        Features DataFrame
    keep_mask : np.ndarray
        Boolean mask of ROIs to keep
    suite2p_path : str or Path
        Path to suite2p/plane0 folder
    """
    suite2p_path = Path(suite2p_path)
    
    # Paths
    iscell_path = suite2p_path / 'iscell.npy'
    iscell_og_path = suite2p_path / 'iscell_og.npy'
    
    # Check if iscell_og already exists
    if iscell_og_path.exists():
        print(f"⚠ Warning: iscell_og.npy already exists!")
        overwrite = input("Overwrite existing iscell_og.npy? (yes/no): ")
        if overwrite.strip().lower() != 'yes':
            print("Operation cancelled.")
            return
    
    # Create updated iscell array
    iscell_updated = data['iscell'].copy()
    
    # Get indices of ROIs to reject (from features_df)
    reject_indices = features_df[~keep_mask]['roi_index'].values.astype(int)
    
    # Set rejected ROIs to 0
    iscell_updated[reject_indices, 0] = 0
    
    # Save original as iscell_og.npy
    np.save(iscell_og_path, data['iscell'])
    print(f"✓ Original saved as: {iscell_og_path}")
    
    # Save updated as iscell.npy
    np.save(iscell_path, iscell_updated)
    print(f"✓ Updated saved as: {iscell_path}")
    
    # Print summary
    n_original_cells = np.sum(data['iscell'][:, 0] == 1)
    n_updated_cells = np.sum(iscell_updated[:, 0] == 1)
    n_removed = n_original_cells - n_updated_cells
    
    print(f"\n{'='*70}")
    print("SAVE COMPLETE")
    print(f"{'='*70}")
    print(f"Original good cells (iscell==1):  {n_original_cells}")
    print(f"Updated good cells (iscell==1):   {n_updated_cells}")
    print(f"ROIs removed:                     {n_removed} ({100*n_removed/n_original_cells:.1f}%)")
    print(f"{'='*70}")
    print(f"\nTo compare in Suite2p GUI:")
    print(f"1. Open Suite2p and load your data")
    print(f"2. Current view shows: iscell.npy (filtered version)")
    print(f"3. To see original: rename iscell_og.npy → iscell.npy")
    print(f"4. Toggle between versions to see what was removed")
    print(f"{'='*70}")


# ============================================================================
# FINAL CHECK AND SAVE
# ============================================================================

# Show final summary
print("="*70)
print("FINAL SUMMARY WITH CURRENT THRESHOLDS")
print("="*70)
print("\nCurrent thresholds:")
for key, value in thresholds.items():
    print(f"  {key}: {value}")

thresholds = {
    'kurtosis_min': 1.0,           # Discard flat/noisy traces (dFF Fisher kurtosis < 1.0)
    'npix_min': 5,                 # Not too small
    'npix_max': 250,               # Not too large (might be merged)
}

keep_mask, rejection_reasons = apply_axon_filters(features_df, thresholds)
print_filter_summary(features_df, keep_mask, rejection_reasons)

# # Check elongated rejections one more time
# print("\n\nChecking elongated rejections with new thresholds...")
# elongated_rejects = diagnose_elongated_rejections(features_df, keep_mask, rejection_reasons, 
#                                                    elongation_threshold=0.5)

# Show the overlay figures
print("\n\nFigure 1: Before vs After Filtering")
print("="*70)
plot_all_rois_overlay(data, features_df, keep_mask)

print("\n\nFigure 2: Rejected ROIs Only (in RED)")
print("="*70)
plot_rejected_rois_overlay(data, features_df, keep_mask)

save_iscell_with_backup(data, features_df, keep_mask, suite2p_path)
print("\n✓ DONE! Open Suite2p GUI to review the filtered ROIs.")
